In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

In [2]:
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [3]:
# Tools 
@tool()
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

In [4]:
tools = [add]
llm_with_tools = llm.bind_tools(tools)

#query = "What is 11 + 49?"
query = "What is the capital of Australia?"

messages = [SystemMessage('You can use tools when necessary, otherwise answer the question using your own knowledge.'),
            HumanMessage(query)]

In [5]:
ai_msg = llm_with_tools.invoke(messages)
print(ai_msg)

content='Canberra.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a16dfe-ef35-7611-90f1-fc3c8a63984a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 70, 'output_tokens': 3, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}}


In [6]:
# Add AI message to messages
messages.append(ai_msg)
# If tool call is present then call the tool and pass response to llm
if len(ai_msg.tool_calls) > 0:
    tool_call = ai_msg.tool_calls[-1]
    selected_tool = eval(tool_call['name'])  # Get function name to call

    # call the tool with required arguments provided by LLM in tool_call
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)  # ToolMessage

    # Provide answer from tool back to LLM
    ai_message = llm_with_tools.invoke(messages)
    messages.append(ai_message)

In [7]:
for message in messages:
    message.pretty_print()

================================ System Message ================================

You can use tools when necessary, otherwise answer the question using your own knowledge.
================================ Human Message =================================

What is the capital of Australia?
================================== Ai Message ==================================

Canberra.
